In [1]:
# I'm importing the BigQuery library to interact with my BigQuery project.
from google.cloud import bigquery

# I'll also import the Pandas library, which is my main tool for data manipulation.
import pandas as pd

# Now, I'll create my main connection to BigQuery.
client = bigquery.Client()

# Finally, I'll load the BigQuery 'magic' command for running simple SQL checks.
%load_ext google.cloud.bigquery


/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/__init__.py:239: FutureWarning: %load_ext google.cloud.bigquery is deprecated. Install bigquery-magics package and use `%load_ext bigquery_magics`, instead.
  warnings.warn(


In [2]:
%%bigquery
-- I'm creating a new dataset named 'fraud_detection_data' to hold all my tables.
CREATE SCHEMA IF NOT EXISTS fraud_detection_data
  OPTIONS(
    location="us-central1",
    description="My dataset for the fraud detection lab"
  );


Query is running:   0%|          |

""


In [3]:
# I'll use the !gsutil cp command to copy the file from the lab's GCS bucket.
# The PDF confirms the path is: gs://labs.roitraining.com/data-to-ai-workshop/fraud_data_raw.csv
!gsutil cp gs://labs.roitraining.com/data-to-ai-workshop/fraud_data_raw.csv fraud_data_raw.csv


Copying gs://labs.roitraining.com/data-to-ai-workshop/fraud_data_raw.csv...
/ [1 files][  4.0 MiB/  4.0 MiB]                                                
Operation completed over 1 objects/4.0 MiB.                                      


In [4]:
# I'm setting up the instructions for the upload.
# My new table will be called 'raw_fraud_data'.
table_id = "fraud_detection_data.raw_fraud_data"
file_path = "fraud_data_raw.csv"

# I'm telling BigQuery it's a CSV and asking it to figure out the columns for me (autodetect=True).
# I'm also telling it to skip the first row because it's a header.
job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    autodetect=True,
    skip_leading_rows=1,
)

# Now, I'll open my local file and start the upload job.
with open(file_path, "rb") as source_file:
    load_job = client.load_table_from_file(source_file, table_id, job_config=job_config)

# I need to wait for the job to complete before I can continue.
load_job.result()

# I'll print a confirmation message so I know the upload was successful.
print(f"I have successfully loaded {load_job.output_rows} rows into the table: {table_id}")


I have successfully loaded 50000 rows into the table: fraud_detection_data.raw_fraud_data


In [5]:
# I'll write a SQL query to select all data from my raw table.
sql_query = """
SELECT *
FROM `fraud_detection_data.raw_fraud_data`
"""

# Now, I'll use the BigQuery client to run that query and load the results into a DataFrame.
df = client.query(sql_query).to_dataframe()

# I'll display the first few rows to make sure the data loaded correctly.
df.head()


,Applicant_ID,Age,Employment_Status,Income,Number_of_Dependents,Amount_Requested,Previous_Assistance_Received,Previous_Assistance_Date,Supporting_Doc_Verified,Application_Frequency_Last_Year,IP_Address,Device_Type,Application_Date,Fraudulent
0,217,65,Unemployed,28984,4,5872,False,NaT,False,1,156.133.45.45,Mobile,2024-08-18,0
1,226,54,Self-Employed,0,1,6631,False,NaT,False,1,245.13.80.245,Tablet,2024-05-11,0
2,240,26,Self-Employed,64477,5,8612,False,NaT,True,1,213.103.170.95,Mobile,2024-08-14,0
3,252,28,Unemployed,28576,4,2951,False,NaT,True,1,234.179.149.207,Desktop,2024-06-12,0
4,266,43,Employed,44930,5,2324,False,NaT,False,1,66.109.96.227,Mobile,2024-08-16,0


In [6]:
# First, I'll make sure the 'Application_Date' column is a proper datetime format.
df['Application_Date'] = pd.to_datetime(df['Application_Date'])

# a, b, c. Create the 'employee', 'unemployed', and 'retired' columns.
df['employee'] = df['Employment_Status'].apply(lambda x: 1 if x == 'Employed' else 0)
df['unemployed'] = df['Employment_Status'].apply(lambda x: 1 if x == 'Unemployed' else 0)
df['retired'] = df['Employment_Status'].apply(lambda x: 1 if x == 'Retired' else 0)

# d. Create the 'age_group' categories.
# I'll define a function to make the logic clean.
def create_age_group(age):
    if 18 <= age <= 30:
        return '18-30'
    elif 31 <= age <= 45:
        return '31-45'
    elif 46 <= age <= 60:
        return '46-60'
    elif age >= 61:
        return '61+'
    else:
        return None
df['age_group'] = df['Age'].apply(create_age_group)

# e. Extract the day of the week from 'Application_Date'.
# The .dt accessor in Pandas is made for this. (Monday=0, Sunday=6, so I'll add 1)
df['application_day_of_week'] = df['Application_Date'].dt.dayofweek + 1

# Now I'll display the first few rows of my transformed DataFrame to check my work.
print("Transformed DataFrame preview:")
df.head()


Transformed DataFrame preview:


,Applicant_ID,Age,Employment_Status,Income,Number_of_Dependents,Amount_Requested,Previous_Assistance_Received,Previous_Assistance_Date,Supporting_Doc_Verified,Application_Frequency_Last_Year,IP_Address,Device_Type,Application_Date,Fraudulent,employee,unemployed,retired,age_group,application_day_of_week
0,217,65,Unemployed,28984,4,5872,False,NaT,False,1,156.133.45.45,Mobile,2024-08-18,0,0,1,0,61+,7
1,226,54,Self-Employed,0,1,6631,False,NaT,False,1,245.13.80.245,Tablet,2024-05-11,0,0,0,0,46-60,6
2,240,26,Self-Employed,64477,5,8612,False,NaT,True,1,213.103.170.95,Mobile,2024-08-14,0,0,0,0,18-30,3
3,252,28,Unemployed,28576,4,2951,False,NaT,True,1,234.179.149.207,Desktop,2024-06-12,0,0,1,0,18-30,3
4,266,43,Employed,44930,5,2324,False,NaT,False,1,66.109.96.227,Mobile,2024-08-16,0,1,0,0,31-45,5


In [7]:
# I'm defining the name for my final, clean table.
final_table_id = "fraud_detection_data.fraud_features_training"

# I'm setting up the job configuration. 'WRITE_TRUNCATE' means it will overwrite the table if it already exists.
job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")

# Now, I'll start the job to upload my Pandas DataFrame to the new BigQuery table.
load_job = client.load_table_from_dataframe(df, final_table_id, job_config=job_config)

# I'll wait for the job to complete.
load_job.result()

# Finally, I'll print a confirmation message.
print(f"Successfully created the final training table: {final_table_id}")


Successfully created the final training table: fraud_detection_data.fraud_features_training
